# Install libraries


In [ ]:
!pip install torch diffusers transformers accelerate gradio safetensors pillow

# Imports libraries

In [ ]:
import torch
import gradio as gr
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler
from PIL import Image
import io
import gc
import random
import tempfile
import os

# 1. Environment Initialization

In [ ]:
runtime_device = "cuda" if torch.cuda.is_available() else "cpu"
precision_type = torch.float16 if runtime_device == "cuda" else torch.float32
print(f"Running on: {runtime_device} | Data type: {precision_type}")

Running on: cpu | Data type: torch.float32



# 2. Custom VAE Loader


In [ ]:
def load_stable_vae(location="madebyollin/sdxl-vae-fp16-fix"):
    """Loads the patched VAE and keeps it in optimal memory state."""
    vae_component = AutoencoderKL.from_pretrained(
        location,
        torch_dtype=precision_type
    )
    # Initially place VAE on CPU – the offloader will handle later
    return vae_component.to("cpu")

print("Acquiring VAE encoder...")
vae_encoder = load_stable_vae()

# 3. Core Pipeline Builder


In [ ]:
base_repo = "stabilityai/stable-diffusion-xl-base-1.0"

print("Assembling generation pipeline...")
pipeline_core = StableDiffusionXLPipeline.from_pretrained(
    base_repo,
    vae=vae_encoder,
    torch_dtype=precision_type,
    use_safetensors=True,
    variant="fp16" if runtime_device == "cuda" else None,
    safety_checker=None,
    requires_safety_checker=False
)

# Replace default scheduler with a faster one (DPMSolver) – reduces steps while keeping quality
pipeline_core.scheduler = DPMSolverMultistepScheduler.from_config(
    pipeline_core.scheduler.config,
    use_karras_sigmas=True,
    algorithm_type="sde-dpmsolver++"
)

# 4. Intelligent Memory Offloading (unique method)


In [ ]:
if runtime_device == "cuda":
    # This moves models to GPU only during inference, saving VRAM
    pipeline_core.enable_model_cpu_offload()
    pipeline_core.enable_attention_slicing()
    print("Memory offloading activated (GPU RAM saver).")
else:
    pipeline_core = pipeline_core.to(runtime_device)

print("All components ready.\n")

All components ready.



# 5. Unique Generator Function with Seed Control & Auto-Fallback


In [ ]:
def create_image_from_text(
    user_prompt: str,
    unwanted_content: str = "blurry, ugly, low quality, distorted, bad anatomy, watermark",
    iteration_count: int = 28,          # not "steps"
    creativity_level: float = 7.2,      # guidance scale
    output_width: int = 1024,
    output_height: int = 1024,
    random_seed: int = None
) -> str:
    """
    Produces a high‑resolution image, saves it as a PNG file,
    and returns the file path for Gradio to serve.
    Includes automatic VRAM recovery and dimension downgrade on OOM.
    """
    # Seed for reproducibility
    if random_seed is None:
        random_seed = random.randint(0, 2**32 - 1)
    generator = torch.Generator(device=runtime_device).manual_seed(random_seed)
    print(f"Generating with seed {random_seed}: '{user_prompt[:60]}...'")

    # Garbage collection before run
    if runtime_device == "cuda":
        torch.cuda.empty_cache()
        gc.collect()

    try:
        with torch.inference_mode():
            result_image = pipeline_core(
                prompt=user_prompt,
                negative_prompt=unwanted_content,
                num_inference_steps=iteration_count,
                guidance_scale=creativity_level,
                width=output_width,
                height=output_height,
                guidance_rescale=0.7,        # prevents colour burning
                generator=generator
            ).images[0]
    except RuntimeError as error:
        if "out of memory" in str(error).lower():
            print("VRAM limit reached – retrying with 768x768 resolution...")
            torch.cuda.empty_cache()
            gc.collect()
            result_image = pipeline_core(
                prompt=user_prompt,
                negative_prompt=unwanted_content,
                num_inference_steps=iteration_count,
                guidance_scale=creativity_level,
                width=768,
                height=768,
                guidance_rescale=0.7,
                generator=generator
            ).images[0]
        else:
            raise error

    # Ensure valid RGB
    if result_image.mode != "RGB":
        result_image = result_image.convert("RGB")

    # --- Save as PNG file ---
    # Create a temporary file with .png extension (will not be deleted automatically)
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".png")
    result_image.save(temp_file.name, format="PNG")
    temp_file.close()

    # Report final size
    file_kb = os.path.getsize(temp_file.name) / 1024
    print(f"PNG saved at: {temp_file.name} | size: {file_kb:.1f} KB")
    return temp_file.name


# 6. Gradio Interface

In [ ]:
demo_examples = [
    "a serene mountain lake at golden hour, highly detailed, 4k, cinematic lighting",
    "a futuristic cyborg chef cooking in a neon kitchen, digital art, trending on artstation",
    "a mystical forest with glowing mushrooms and a tiny fox, fantasy illustration, deep colours",
    "an abstract explosion of geometric shapes and vibrant gradients, 8k, sharp"
]

interface = gr.Interface(
    fn=create_image_from_text,
    inputs=[
        gr.Textbox(label="What do you want to see?", placeholder="Describe your image...", value=demo_examples[0]),
        gr.Textbox(label="Avoid these elements", value="blurry, ugly, low quality, distorted, bad anatomy, watermark"),
        gr.Slider(minimum=15, maximum=50, step=1, label="Sampling iterations", value=28),
        gr.Slider(minimum=2.0, maximum=15.0, step=0.5, label="Creative freedom (guidance)", value=7.2),
        gr.Slider(minimum=512, maximum=1024, step=64, label="Output width", value=1024),
        gr.Slider(minimum=512, maximum=1024, step=64, label="Output height", value=1024),
        gr.Number(label="Random seed (optional – leave empty for random)", value=None)
    ],
    outputs=gr.Image(label="Generated artwork (PNG)", type="filepath"),  # ← changed to filepath
    title="AI Text‑to‑Image Generator (SDXL",
    description="Original SDXL Image Forge – High‑Resolution Text‑to‑Image Generator",
    examples=[[ex] for ex in demo_examples]
)

# 7. Launch

In [ ]:
if __name__ == "__main__":
    interface.launch(share=True)